# Análise Exploratória e Diagnóstico de Qualidade (EDA)
Neste notebook, exploramos as tabelas originais do Olist (Camada Raw) para identificar inconsistências, valores nulos e duplicatas. O objetivo é fundamentar as regras de limpeza que aplicaremos na nossa **Camada Staging**.

In [ ]:
import pandas as pd
import glob
import os

# Carrega os datasets
paths = glob.glob("../data/raw/*.csv")
dfs = {os.path.basename(p).replace(".csv", ""): pd.read_csv(p) for p in paths}
print(f"{len(dfs)} tabelas carregadas.")

## 1. Visão Geral: Nulos e Duplicatas
Um loop por todas as tabelas para mapear o cenário macro.

In [ ]:
for nome, df in dfs.items():
    print(f"\n=== {nome} | shape={df.shape} ===")
    nulos = (df.isnull().mean() * 100).round(2)
    print("Nulos (%):")
    print(nulos[nulos > 0].to_dict() if (nulos > 0).any() else "{}")
    print("Duplicatas:", df.duplicated().sum())

## 2. Diagnóstico: Pedidos (`olist_orders_dataset`)
Existem nulos em datas importantes de entrega. Como nosso projeto foca em **predição de atraso**, não podemos usar pedidos sem data de entrega.

In [ ]:
orders = dfs["olist_orders_dataset"]
# Filtrar apenas pedidos Entregues
entregues = orders[orders["order_status"] == "delivered"]
print(f"Total de pedidos originais: {len(orders)}")
print(f"Total de pedidos entregues: {len(entregues)}")
print(f"Nulos na data de entrega ao cliente (entre os Entregues): {entregues['order_delivered_customer_date'].isnull().sum()}")

> **Conclusão:** Em Analytics, usaremos um filtro (`WHERE order_status = 'delivered' AND delivered_ts IS NOT NULL`).

## 3. Diagnóstico: Avaliações (`olist_order_reviews_dataset`)
A grande maioria não deixa comentários. E precisamos verificar a cardinalidade (1 pedido = 1 review?).

In [ ]:
reviews = dfs["olist_order_reviews_dataset"]
print("Total de avaliações:", len(reviews))
print("Pedidos únicos com avaliação:", reviews["order_id"].nunique())
print(f"Pedidos com mais de uma avaliação: {len(reviews) - reviews['order_id'].nunique()}")

> **Conclusão:** Existem pedidos com mais de um review. Na Staging, usaremos `DISTINCT ON (order_id)` pegando a mais recente para manter o grão de 1 linha = 1 pedido.

## 4. Diagnóstico: Geolocalização (`olist_geolocation_dataset`)
É a tabela com maior risco de causar explosão de linhas nos Joins (Produto Cartesiano).

In [ ]:
geo = dfs["olist_geolocation_dataset"]
print("Total de linhas:", len(geo))
print("Duplicatas exatas:", geo.duplicated().sum())
print("Prefixos de CEP únicos:", geo["geolocation_zip_code_prefix"].nunique()")

> **Conclusão:** Temos mais de 1 milhão de linhas, mas apenas ~19 mil CEPs únicos. Na Staging, precisamos agrupar (`GROUP BY`) por prefixo de CEP e tirar a média das coordenadas para desduplicar.